# Claim Occurrence Modeling

## Objective

The goal of this notebook is to predict whether an Auto insurance contract will have a claim during its contract period.

The modeling dataset comes from the `vw_auto_claim_occurrence_ml` SQL view.

Two feature sets will be compared:

- Core features
- Core features with selected engineered features

The target variable is `has_claim`.

Because claim occurrence is rare, PR-AUC will be the main model-selection metric.

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
)

from sklearn.base import clone
from sklearn.model_selection import (StratifiedKFold, GridSearchCV, 
                                     train_test_split, RepeatedStratifiedKFold, 
                                     cross_validate)
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)

from xgboost import XGBClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    brier_score_loss
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
conn = pyodbc.connect(
    "DSN=InsuranceAnalytics;",
    autocommit=True
)

query = """
SELECT *
FROM vw_auto_claim_occurrence_ml
"""

df = pd.read_sql(query, conn)

df["start_date"] = pd.to_datetime(df["start_date"])
df["end_date"] = pd.to_datetime(df["end_date"])

print("Shape:", df.shape)
print("Claims:", df["has_claim"].sum())
print("Claim rate:", f"{df['has_claim'].mean() * 100:.2f}%")

Shape: (4327, 20)
Claims: 88
Claim rate: 2.03%


## Deterministic Feature Engineering

New features are created only from information available before a claim occurs.

These transformations do not learn anything from the target or from the test set.

In [3]:
## deterministic feature engineering

model_df = df.copy()

model_df["vehicle_age"] = (
    model_df["start_date"].dt.year
    - model_df["year"]
).clip(lower=0)

model_df["contract_duration_days"] = (
    model_df["end_date"]
    - model_df["start_date"]
).dt.days

model_df["premium_value_ratio"] = (
    model_df["annual_premium"]
    / model_df["current_value"]
)

model_df["vehicle_age_group"] = pd.cut(
    model_df["vehicle_age"],
    bins=[-1, 3, 7, np.inf],
    labels=["0-3", "4-7", "8+"],
).astype("object")

model_df["client_age_group"] = pd.cut(
    model_df["client_age"],
    bins=[17, 29, 39, 49, 59, np.inf],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60+",
    ],
).astype("object")

model_df["previous_claims_cat"] = (
    model_df["previous_claims"]
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

In [4]:
core_numeric_features = [
    "annual_premium",
    "client_age",
    "power_hp",
    "current_value",
    "vehicle_age",
]

core_categorical_features = [
    "risk_zone",
    "channel",
    "csp",
    "gender",
    "city",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims_cat",
]

CORE_FEATURES = (
    core_numeric_features
    + core_categorical_features
)

In [5]:
engineered_numeric_features = [
    "contract_duration_days",
    "premium_value_ratio",
]

engineered_categorical_features = [
    "vehicle_age_group",
    "client_age_group",
]

EXTENDED_FEATURES = (
    CORE_FEATURES
    + engineered_numeric_features
    + engineered_categorical_features
)

In [6]:
TARGET = "has_claim"

X_core = model_df[
    CORE_FEATURES
].copy()

X_extended = model_df[
    EXTENDED_FEATURES
].copy()

y = model_df[
    TARGET
].copy()

print("Core shape:", X_core.shape)
print("Extended shape:", X_extended.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Core shape: (4327, 14)
Extended shape: (4327, 18)
Target shape: (4327,)

Target distribution:
has_claim
0    4239
1      88
Name: count, dtype: int64


In [7]:
all_indices = np.arange(
    len(model_df)
)

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

In [8]:
X_train_core = X_core.iloc[
    train_idx
].copy()

X_test_core = X_core.iloc[
    test_idx
].copy()

X_train_extended = X_extended.iloc[
    train_idx
].copy()

X_test_extended = X_extended.iloc[
    test_idx
].copy()

y_train = y.iloc[
    train_idx
].copy()

y_test = y.iloc[
    test_idx
].copy()

In [10]:
split_summary = pd.DataFrame(
    {
        "dataset": [
            "Train",
            "Test",
        ],
        "contracts": [
            len(y_train),
            len(y_test),
        ],
        "claims": [
            y_train.sum(),
            y_test.sum(),
        ],
        "claim_rate_pct": [
            y_train.mean() * 100,
            y_test.mean() * 100,
        ],
    }
)

split_summary

,dataset,contracts,claims,claim_rate_pct
0,Train,3461,70,2.0225
1,Test,866,18,2.0785


## Train-Based Preprocessing Decisions

All preprocessing decisions are made using only the training data.

This prevents information from the test set from affecting the modeling process.

We will first check numeric correlations and outliers before selecting the scaling strategy.

In [11]:
extended_numeric_features = (
    core_numeric_features
    + engineered_numeric_features
)

train_numeric = X_train_extended[
    extended_numeric_features
].copy()

In [12]:
train_corr = (
    train_numeric
    .corr(method="spearman")
    .round(3)
)

train_corr

,annual_premium,client_age,power_hp,current_value,vehicle_age,contract_duration_days,premium_value_ratio
annual_premium,1.0000,-0.0960,-0.0000,0.3520,-0.2730,-0.0030,0.1190
client_age,-0.0960,1.0000,0.0080,0.2510,-0.2760,-0.0080,-0.3010
power_hp,-0.0000,0.0080,1.0000,0.0300,-0.0220,-0.0070,-0.0300
current_value,0.3520,0.2510,0.0300,1.0000,-0.8450,-0.0200,-0.8600
vehicle_age,-0.2730,-0.2760,-0.0220,-0.8450,1.0000,0.0020,0.7460
contract_duration_days,-0.0030,-0.0080,-0.0070,-0.0200,0.0020,1.0000,0.0190
premium_value_ratio,0.1190,-0.3010,-0.0300,-0.8600,0.7460,0.0190,1.0000


In [13]:
corr_abs = train_corr.abs()

upper_triangle = corr_abs.where(
    np.triu(
        np.ones(corr_abs.shape),
        k=1,
    ).astype(bool)
)

high_corr_pairs = []

for column in upper_triangle.columns:
    for index in upper_triangle.index:
        value = upper_triangle.loc[index, column]

        if pd.notna(value) and value >= 0.80:
            high_corr_pairs.append(
                {
                    "feature_1": index,
                    "feature_2": column,
                    "abs_spearman_corr": value,
                }
            )

high_corr_pairs = pd.DataFrame(
    high_corr_pairs
)

high_corr_pairs

,feature_1,feature_2,abs_spearman_corr
0,current_value,vehicle_age,0.8450
1,current_value,premium_value_ratio,0.8600


In [14]:
def outlier_summary_iqr(
    df,
    columns,
    multiplier=1.5,
):
    results = []

    for column in columns:
        values = df[column].dropna()

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - multiplier * iqr
        upper_bound = q3 + multiplier * iqr

        outlier_mask = (
            (values < lower_bound)
            | (values > upper_bound)
        )

        outlier_count = outlier_mask.sum()

        results.append(
            {
                "feature": column,
                "q1": q1,
                "q3": q3,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "outlier_count": outlier_count,
                "outlier_pct": (
                    outlier_count
                    / len(values)
                    * 100
                ),
            }
        )

    return pd.DataFrame(results)

In [15]:
outlier_summary = outlier_summary_iqr(
    X_train_extended,
    extended_numeric_features,
)

outlier_summary

,feature,q1,q3,lower_bound,upper_bound,outlier_count,outlier_pct
0,annual_premium,657.3900,"1,005.9100",134.6100,"1,528.6900",27,0.7801
1,client_age,35.0000,52.0000,9.5000,77.5000,0,0.0000
2,power_hp,117.0000,172.6100,33.5850,256.0250,0,0.0000
3,current_value,"5,385.1400","12,892.5900","-5,876.0350","24,153.7650",150,4.3340
4,vehicle_age,2.0000,6.0000,-4.0000,12.0000,160,4.8721
5,contract_duration_days,346.0000,382.0000,292.0000,436.0000,0,0.0000
6,premium_value_ratio,0.0647,0.1323,-0.0368,0.2338,111,3.2072


### Correlation and Outlier Decisions

The training data does not contain numeric feature pairs with an absolute correlation above 0.90.

`current_value` is strongly related to `vehicle_age` and `premium_value_ratio`. However, these variables represent different risk information and will remain as candidate features.

`current_value`, `vehicle_age`, and `premium_value_ratio` contain more outliers than the other numeric variables. These features will use RobustScaler.

The other numeric features will use StandardScaler.

No outliers will be removed because they may represent valid insurance contracts.

In [16]:
standard_transformer = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "standard_scaler",
            StandardScaler(),
        ),
    ]
)

robust_transformer = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "robust_scaler",
            RobustScaler(),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "unknown_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

In [17]:
core_standard_columns = [
    "annual_premium",
    "client_age",
    "power_hp",
]

core_robust_columns = [
    "current_value",
    "vehicle_age",
]

core_onehot_columns = [
    "risk_zone",
    "channel",
    "csp",
    "gender",
    "city",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims_cat",
]

In [18]:
core_preprocess = ColumnTransformer(
    transformers=[
        (
            "standard_num",
            standard_transformer,
            core_standard_columns,
        ),
        (
            "robust_num",
            robust_transformer,
            core_robust_columns,
        ),
        (
            "categorical",
            categorical_transformer,
            core_onehot_columns,
        ),
    ],
    remainder="drop",
)

In [19]:
extended_standard_columns = [
    "annual_premium",
    "client_age",
    "power_hp",
    "contract_duration_days",
]

extended_robust_columns = [
    "current_value",
    "vehicle_age",
    "premium_value_ratio",
]

extended_onehot_columns = [
    "risk_zone",
    "channel",
    "csp",
    "gender",
    "city",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims_cat",
    "vehicle_age_group",
    "client_age_group",
]

In [20]:
extended_preprocess = ColumnTransformer(
    transformers=[
        (
            "standard_num",
            standard_transformer,
            extended_standard_columns,
        ),
        (
            "robust_num",
            robust_transformer,
            extended_robust_columns,
        ),
        (
            "categorical",
            categorical_transformer,
            extended_onehot_columns,
        ),
    ],
    remainder="drop",
)

In [21]:
core_preprocessing_columns = (
    core_standard_columns
    + core_robust_columns
    + core_onehot_columns
)

extended_preprocessing_columns = (
    extended_standard_columns
    + extended_robust_columns
    + extended_onehot_columns
)

print("CORE")
print("X features:", len(X_train_core.columns))
print(
    "Missing:",
    set(X_train_core.columns)
    - set(core_preprocessing_columns),
)
print(
    "Unexpected:",
    set(core_preprocessing_columns)
    - set(X_train_core.columns),
)

print("\nEXTENDED")
print("X features:", len(X_train_extended.columns))
print(
    "Missing:",
    set(X_train_extended.columns)
    - set(extended_preprocessing_columns),
)
print(
    "Unexpected:",
    set(extended_preprocessing_columns)
    - set(X_train_extended.columns),
)

CORE
X features: 14
Missing: set()
Unexpected: set()

EXTENDED
X features: 18
Missing: set()
Unexpected: set()


## Core vs Extended Feature Set Comparison

The core and extended feature sets will be compared using the same models and the same cross-validation splits.

Repeated stratified cross-validation is used because the positive class is rare.

PR-AUC is the main metric.

ROC-AUC is also reported to measure ranking performance.

The extended feature set will only be kept if it provides a consistent improvement.

In [22]:
repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

In [23]:
base_models = {
    "LR": LogisticRegression(
        max_iter=5000,
        random_state=42,
    ),

    "LR_Balanced": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=15,
    ),

    "SVC": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight=None,
    ),

    "SVC_Balanced": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
    ),

    "DecisionTree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        random_state=42,
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),

    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=2,
        random_state=42,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    ),
}

In [24]:
feature_sets = {
    "Core": {
        "X": X_train_core,
        "preprocess": core_preprocess,
    },

    "Extended": {
        "X": X_train_extended,
        "preprocess": extended_preprocess,
    },
}

In [26]:
comparison_results = []

for feature_set_name, config in feature_sets.items():

    X_current = config["X"]
    preprocess_current = config["preprocess"]

    for model_name, estimator in base_models.items():

        model_pipeline = Pipeline(
            steps=[
                (
                    "preprocess",
                    preprocess_current,
                ),
                (
                    "model",
                    estimator,
                ),
            ]
        )

        scores = cross_validate(
            model_pipeline,
            X_current,
            y_train,
            cv=repeated_cv,
            scoring={
                "pr_auc": "average_precision",
                "roc_auc": "roc_auc",
            },
            n_jobs=-1,
            return_train_score=False,
        )

        comparison_results.append(
            {
                "feature_set": feature_set_name,
                "model": model_name,

                "pr_auc_mean": (
                    scores["test_pr_auc"].mean()
                ),
                "pr_auc_std": (
                    scores["test_pr_auc"].std()
                ),
                "pr_auc_median": (
                    np.median(
                        scores["test_pr_auc"]
                    )
                ),

                "roc_auc_mean": (
                    scores["test_roc_auc"].mean()
                ),
                "roc_auc_std": (
                    scores["test_roc_auc"].std()
                ),
            }
        )

comparison_results = pd.DataFrame(
    comparison_results
)

In [27]:
comparison_results.sort_values(
    "pr_auc_mean",
    ascending=False,
)

,feature_set,model,pr_auc_mean,pr_auc_std,pr_auc_median,roc_auc_mean,roc_auc_std
13,Extended,SVC,0.0244,0.0073,0.0224,0.4826,0.0705
12,Extended,KNN,0.0239,0.0063,0.0214,0.5272,0.0552
15,Extended,DecisionTree,0.0235,0.0054,0.0220,0.4877,0.0907
8,Core,GradientBoosting,0.0231,0.0091,0.0201,0.4303,0.0702
16,Extended,RandomForest,0.0231,0.0059,0.0227,0.4491,0.0723
11,Extended,LR_Balanced,0.0230,0.0117,0.0180,0.3987,0.0765
18,Extended,GradientBoosting,0.0229,0.0054,0.0213,0.4573,0.0486
14,Extended,SVC_Balanced,0.0228,0.0059,0.0207,0.4554,0.0667
10,Extended,LR,0.0223,0.0091,0.0192,0.3886,0.0790
0,Core,LR,0.0222,0.0082,0.0206,0.3987,0.0743


In [28]:
feature_set_comparison = (
    comparison_results
    .pivot(
        index="model",
        columns="feature_set",
        values="pr_auc_mean",
    )
)

feature_set_comparison["improvement"] = (
    feature_set_comparison["Extended"]
    - feature_set_comparison["Core"]
)

feature_set_comparison.sort_values(
    "improvement",
    ascending=False,
)

feature_set,Core,Extended,improvement
model,,,
SVC,0.0192,0.0244,0.0052
ExtraTrees,0.0176,0.0219,0.0042
SVC_Balanced,0.0193,0.0228,0.0035
KNN,0.0208,0.0239,0.0030
RandomForest,0.0205,0.0231,0.0026
DecisionTree,0.0213,0.0235,0.0022
XGBoost,0.0196,0.0213,0.0016
LR_Balanced,0.0215,0.0230,0.0015
LR,0.0222,0.0223,0.0001


### Feature Set Comparison Results

The extended feature set improves PR-AUC for most of the tested models.

This suggests that the engineered features add some useful information to the original dataset.

However, the overall predictive performance is still limited.

KNN gives the most balanced result. It has a PR-AUC above the positive-class baseline and the highest ROC-AUC among the extended models.

For this reason, the extended feature set will be used for the next modeling steps.

In [29]:
knn_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            extended_preprocess,
        ),
        (
            "model",
            KNeighborsClassifier(),
        ),
    ]
)

In [30]:
knn_param_grid = {
    "model__n_neighbors": [
        5,
        10,
        15,
        20,
        25,
        30,
        40,
        50,
    ],
    "model__weights": [
        "uniform",
        "distance",
    ],
    "model__p": [
        1,
        2,
    ],
}

In [31]:
knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    scoring={
        "pr_auc": "average_precision",
        "roc_auc": "roc_auc",
    },
    refit="pr_auc",
    cv=repeated_cv,
    n_jobs=-1,
    return_train_score=True,
)

knn_grid.fit(
    X_train_extended,
    y_train,
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': [5, 10, ...], 'model__p': [1, 2], 'model__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'pr_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed fro

In [32]:
print("Best parameters:")
print(knn_grid.best_params_)

print(
    "\nBest CV PR-AUC:",
    round(knn_grid.best_score_, 4),
)

Best parameters:
{'model__n_neighbors': 10, 'model__p': 2, 'model__weights': 'distance'}

Best CV PR-AUC: 0.0328


In [33]:
knn_tuning_results = (
    pd.DataFrame(knn_grid.cv_results_)
    [
        [
            "param_model__n_neighbors",
            "param_model__weights",
            "param_model__p",
            "mean_train_pr_auc",
            "mean_test_pr_auc",
            "std_test_pr_auc",
            "mean_test_roc_auc",
        ]
    ]
    .sort_values(
        "mean_test_pr_auc",
        ascending=False,
    )
    .head(10)
)

knn_tuning_results

,param_model__n_neighbors,param_model__weights,param_model__p,mean_train_pr_auc,mean_test_pr_auc,std_test_pr_auc,mean_test_roc_auc
7,10,distance,2,1.0000,0.0328,0.0171,0.5200
3,5,distance,2,1.0000,0.0325,0.0262,0.4993
5,10,distance,1,1.0000,0.0290,0.0128,0.5120
11,15,distance,2,1.0000,0.0288,0.0116,0.5242
9,15,distance,1,1.0000,0.0279,0.0116,0.5105
31,50,distance,2,1.0000,0.0270,0.0153,0.4688
15,20,distance,2,1.0000,0.0266,0.0085,0.5129
6,10,uniform,2,0.1528,0.0261,0.0088,0.5223
27,40,distance,2,1.0000,0.0261,0.0082,0.4777
19,25,distance,2,1.0000,0.0255,0.0072,0.5070


### KNN Tuning Results

Hyperparameter tuning improves the KNN model.

The highest PR-AUC is obtained with:

- `n_neighbors = 10`
- `weights = distance`
- `p = 2`

The mean PR-AUC increases to about 0.033.

However, the model still shows some variation across cross-validation folds.

Distance-weighted KNN also has a very high training score because each training observation is very close to itself. Therefore, training PR-AUC is not used for model selection.

A stability check will be performed before the final model is selected.

In [34]:
knn_candidates = {
    "KNN_Best_PR": KNeighborsClassifier(
        n_neighbors=10,
        weights="distance",
        p=2,
    ),

    "KNN_Stable": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance",
        p=2,
    ),

    "KNN_Uniform": KNeighborsClassifier(
        n_neighbors=10,
        weights="uniform",
        p=2,
    ),
}

In [35]:
stability_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=123,
)

In [36]:
knn_stability_results = []

for model_name, estimator in knn_candidates.items():

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                extended_preprocess,
            ),
            (
                "model",
                estimator,
            ),
        ]
    )

    scores = cross_validate(
        model_pipeline,
        X_train_extended,
        y_train,
        cv=stability_cv,
        scoring={
            "pr_auc": "average_precision",
            "roc_auc": "roc_auc",
        },
        n_jobs=-1,
    )

    knn_stability_results.append(
        {
            "model": model_name,

            "pr_auc_mean": (
                scores["test_pr_auc"].mean()
            ),

            "pr_auc_median": np.median(
                scores["test_pr_auc"]
            ),

            "pr_auc_std": (
                scores["test_pr_auc"].std()
            ),

            "roc_auc_mean": (
                scores["test_roc_auc"].mean()
            ),

            "roc_auc_std": (
                scores["test_roc_auc"].std()
            ),
        }
    )

knn_stability_summary = (
    pd.DataFrame(knn_stability_results)
    .sort_values(
        "pr_auc_mean",
        ascending=False,
    )
)

knn_stability_summary

,model,pr_auc_mean,pr_auc_median,pr_auc_std,roc_auc_mean,roc_auc_std
0,KNN_Best_PR,0.0386,0.0317,0.0254,0.5314,0.0534
1,KNN_Stable,0.0334,0.0275,0.0211,0.5374,0.0512
2,KNN_Uniform,0.0276,0.0255,0.0095,0.5344,0.0537


### KNN Stability Results

The tuned KNN models remain better than the positive-class baseline under new cross-validation splits.

The KNN with 10 neighbors has the highest mean and median PR-AUC.

The 15-neighbor model is slightly more stable and has a slightly higher ROC-AUC.

Both models will therefore be evaluated for probability quality before the final model is selected.

In [37]:
probability_candidates = {
    "KNN_10": KNeighborsClassifier(
        n_neighbors=10,
        weights="distance",
        p=2,
    ),

    "KNN_15": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance",
        p=2,
    ),
}

In [38]:
probability_results = []

for fold, (train_idx, val_idx) in enumerate(
    stability_cv.split(
        X_train_extended,
        y_train,
    ),
    start=1,
):
    X_tr = X_train_extended.iloc[train_idx]
    X_val = X_train_extended.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # -----------------------------
    # Prevalence baseline
    # -----------------------------
    train_prevalence = y_tr.mean()

    baseline_proba = np.full(
        len(y_val),
        train_prevalence,
    )

    probability_results.append(
        {
            "model": "Prevalence_Baseline",
            "fold": fold,
            "pr_auc": average_precision_score(
                y_val,
                baseline_proba,
            ),
            "roc_auc": 0.5,
            "brier_score": brier_score_loss(
                y_val,
                baseline_proba,
            ),
            "mean_probability": (
                baseline_proba.mean()
            ),
            "actual_prevalence": (
                y_val.mean()
            ),
        }
    )

    # -----------------------------
    # KNN models
    # -----------------------------
    for model_name, estimator in (
        probability_candidates.items()
    ):
        model_pipeline = Pipeline(
            steps=[
                (
                    "preprocess",
                    extended_preprocess,
                ),
                (
                    "model",
                    estimator,
                ),
            ]
        )

        fold_model = clone(
            model_pipeline
        )

        fold_model.fit(
            X_tr,
            y_tr,
        )

        val_proba = (
            fold_model
            .predict_proba(X_val)[:, 1]
        )

        probability_results.append(
            {
                "model": model_name,
                "fold": fold,

                "pr_auc": average_precision_score(
                    y_val,
                    val_proba,
                ),

                "roc_auc": roc_auc_score(
                    y_val,
                    val_proba,
                ),

                "brier_score": brier_score_loss(
                    y_val,
                    val_proba,
                ),

                "mean_probability": (
                    val_proba.mean()
                ),

                "actual_prevalence": (
                    y_val.mean()
                ),
            }
        )

probability_results = pd.DataFrame(
    probability_results
)

In [39]:
probability_summary = (
    probability_results
    .groupby("model")
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_median=("pr_auc", "median"),
        pr_auc_std=("pr_auc", "std"),

        roc_auc_mean=("roc_auc", "mean"),

        brier_mean=("brier_score", "mean"),
        brier_std=("brier_score", "std"),

        mean_probability=(
            "mean_probability",
            "mean",
        ),

        actual_prevalence=(
            "actual_prevalence",
            "mean",
        ),
    )
    .sort_values(
        "brier_mean",
        ascending=True,
    )
)

probability_summary

,pr_auc_mean,pr_auc_median,pr_auc_std,roc_auc_mean,brier_mean,brier_std,mean_probability,actual_prevalence
model,,,,,,,,
Prevalence_Baseline,0.0202,0.0202,0.0000,0.5000,0.0198,0.0000,0.0202,0.0202
KNN_15,0.0334,0.0275,0.0213,0.5374,0.0210,0.0004,0.0217,0.0202
KNN_10,0.0386,0.0317,0.0257,0.5314,0.0216,0.0006,0.0222,0.0202


### Probability Evaluation

The engineered features improve the risk-ranking performance of the KNN models.

The KNN models have higher PR-AUC and ROC-AUC than the prevalence baseline.

However, their Brier Scores are worse than the simple prevalence benchmark.

This means that KNN provides useful ranking information, but its raw probabilities are not reliable enough for Expected Loss estimation.

The 15-neighbor KNN has slightly better probability quality and stability than the 10-neighbor model.

In [40]:
knn_15_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            extended_preprocess,
        ),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=15,
                weights="distance",
                p=2,
            ),
        ),
    ]
)

In [41]:
calibrated_knn = CalibratedClassifierCV(
    estimator=knn_15_pipeline,
    method="sigmoid",
    cv=3,
)

In [42]:
calibrated_results = []

for fold, (train_idx, val_idx) in enumerate(
    stability_cv.split(
        X_train_extended,
        y_train,
    ),
    start=1,
):
    X_tr = X_train_extended.iloc[train_idx]
    X_val = X_train_extended.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    fold_model = clone(
        calibrated_knn
    )

    fold_model.fit(
        X_tr,
        y_tr,
    )

    val_proba = fold_model.predict_proba(
        X_val
    )[:, 1]

    calibrated_results.append(
        {
            "fold": fold,

            "pr_auc": average_precision_score(
                y_val,
                val_proba,
            ),

            "roc_auc": roc_auc_score(
                y_val,
                val_proba,
            ),

            "brier_score": brier_score_loss(
                y_val,
                val_proba,
            ),

            "mean_probability": (
                val_proba.mean()
            ),

            "actual_prevalence": (
                y_val.mean()
            ),
        }
    )

calibrated_results = pd.DataFrame(
    calibrated_results
)

In [43]:
calibrated_summary = pd.Series(
    {
        "pr_auc_mean": (
            calibrated_results["pr_auc"].mean()
        ),

        "pr_auc_median": (
            calibrated_results["pr_auc"].median()
        ),

        "pr_auc_std": (
            calibrated_results["pr_auc"].std()
        ),

        "roc_auc_mean": (
            calibrated_results["roc_auc"].mean()
        ),

        "brier_mean": (
            calibrated_results["brier_score"].mean()
        ),

        "brier_std": (
            calibrated_results["brier_score"].std()
        ),

        "mean_probability": (
            calibrated_results[
                "mean_probability"
            ].mean()
        ),

        "actual_prevalence": (
            calibrated_results[
                "actual_prevalence"
            ].mean()
        ),
    }
)

calibrated_summary

pr_auc_mean         0.0226
pr_auc_median       0.0208
pr_auc_std          0.0063
roc_auc_mean        0.4729
brier_mean          0.0198
brier_std           0.0000
mean_probability    0.0203
actual_prevalence   0.0202
dtype: float64

### Probability Calibration Result

Sigmoid calibration improves the Brier Score of KNN, but it reduces the ranking performance.

The calibrated model has a Brier Score close to the prevalence baseline, but its ROC-AUC falls below 0.50.

This means that calibration removes much of the useful ranking information.

For this reason, the calibrated KNN will not be used.

The tuned KNN will be kept as an experimental risk-ranking model, while the training-set claim rate will remain the probability benchmark.

In [44]:
final_knn = Pipeline(
    steps=[
        (
            "preprocess",
            extended_preprocess,
        ),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=10,
                weights="distance",
                p=2,
            ),
        ),
    ]
)

final_knn.fit(
    X_train_extended,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['annual_premium','client_age','power_hp',...,'premium_value_ratio', 'vehicle_age_group','client_age_group']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standard_num', ...), ('robust_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'

In [45]:
test_proba = final_knn.predict_proba(
    X_test_extended
)[:, 1]

test_pr_auc = average_precision_score(
    y_test,
    test_proba,
)

test_roc_auc = roc_auc_score(
    y_test,
    test_proba,
)

print(
    "Test PR-AUC:",
    round(test_pr_auc, 4),
)

print(
    "Test ROC-AUC:",
    round(test_roc_auc, 4),
)

print(
    "Test prevalence:",
    round(y_test.mean(), 4),
)

Test PR-AUC: 0.0204
Test ROC-AUC: 0.451
Test prevalence: 0.0208


In [46]:
train_prevalence = y_train.mean()

baseline_test_proba = np.full(
    len(y_test),
    train_prevalence,
)

test_baseline_brier = brier_score_loss(
    y_test,
    baseline_test_proba,
)

knn_test_brier = brier_score_loss(
    y_test,
    test_proba,
)

print(
    "Train prevalence:",
    round(train_prevalence, 4),
)

print(
    "KNN Test Brier:",
    round(knn_test_brier, 4),
)

print(
    "Baseline Test Brier:",
    round(test_baseline_brier, 4),
)

Train prevalence: 0.0202
KNN Test Brier: 0.0231
Baseline Test Brier: 0.0204


## Final Test Evaluation

The engineered feature set improved cross-validation performance, especially for KNN.

However, this improvement did not generalize to the untouched test set.

The final KNN model achieved a PR-AUC of 0.0204, which is close to the test claim rate of 0.0208.

The ROC-AUC was 0.451, which also shows weak ranking performance.

The KNN Brier Score was worse than the simple prevalence benchmark.

Therefore, threshold optimization is not performed. Changing the classification threshold cannot improve the underlying ranking quality of the model.

The current dataset does not provide enough stable information for a reliable individual claim occurrence model.

## Save Train-Test Split

The contract IDs used in the training and test sets are saved for the Expected Loss analysis.

This ensures that the same evaluation population is used across notebooks.

In [50]:
all_indices = np.arange(len(model_df))

holdout_train_idx, holdout_test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

In [51]:
print("Train:", len(holdout_train_idx))
print("Test:", len(holdout_test_idx))

Train: 3461
Test: 866


In [52]:
split_df = model_df[
    ["contract_id"]
].copy()

split_df["dataset"] = "Test"

split_df.iloc[
    holdout_train_idx,
    split_df.columns.get_loc("dataset"),
] = "Train"

In [53]:
split_check = (
    model_df[
        ["contract_id", "has_claim"]
    ]
    .merge(
        split_df,
        on="contract_id",
        how="left",
    )
    .groupby("dataset")
    .agg(
        contracts=("contract_id", "size"),
        claims=("has_claim", "sum"),
        claim_rate=("has_claim", "mean"),
    )
)

split_check["claim_rate_pct"] = (
    split_check["claim_rate"] * 100
)

split_check

,contracts,claims,claim_rate,claim_rate_pct
dataset,,,,
Test,866,18,0.0208,2.0785
Train,3461,70,0.0202,2.0225


In [54]:
from pathlib import Path

output_path = Path(
    "../data/processed/occurrence_split_map.csv"
)

split_df.to_csv(
    output_path,
    index=False,
)

print("Saved:", output_path.resolve())
print("Shape:", split_df.shape)

Saved: C:\Users\okand\Desktop\Projects\insurance-analytics-platform\data\processed\occurrence_split_map.csv
Shape: (4327, 2)


# Expected Loss Modeling

## Objective

The goal of this notebook is to estimate the expected claim cost for an Auto insurance contract.

Expected Loss combines two components:

- claim occurrence probability,
- expected claim severity.

The basic formula is:

Expected Loss = Claim Probability × Expected Severity

The claim occurrence model did not produce reliable individual probabilities on the final test set.

Therefore, the training portfolio claim rate will be used as the initial frequency benchmark.

Claim severity models that use `claim_type` cannot be used directly before a claim occurs because claim type is not known at contract inception.

For this reason, the Expected Loss analysis will use only information available before claim occurrence.